## Source metadata spreadsheet batch converter

This notebook converts metadata from a pared-down SourceMetaCuration spreadsheet into json files for upload into the DDE.

The tabbed sheets available in the spreadsheet are as follows:
- resource_base
- collectionSize
- definedTerms
- spatialTemporalCoverage

All sheets use the url field as the index/linking field

Notes:
* resource_base: contains various metadata properties and their expected values
* spatialTemporalCoverage: maps rows to `spatialCoverage` (`AdministrativeArea`) and `temporalCoverage` (`TemporalInterval`) objects following the NDE ResourceCatalog schema
* definedTerms: save only the urls to a list for the DDE

How it works: 
The resource_base is converted into a base dictionary, and additional objects from collectionSize, definedTerms, and spatialTemporalCoverage are added using the url as the key.
The json records are then dumped into a batch_file for upload.

In [1]:
import os
import pandas as pd
import json
from datetime import datetime
from math import isnan

In [2]:
script_path = os.getcwd()
parent_path = os.path.abspath(os.path.join(script_path, os.pardir))
data_path = os.path.join(script_path,'data')
filelist = os.listdir(data_path)
result_path = os.path.join(parent_path,'nde-metadata-corrections','metadata_for_DDE','resourceCatalogs')

In [3]:
print(parent_path)
print(filelist)

C:\Users\gtsueng\Anaconda3\envs\nde
['2024_05_14_RepoMetaCuration.xlsx', '2024_05_21_RepoMetaCuration.xlsx', '2024_06_18_RepoMetaCuration.xlsx', '2024_06_21_RepoMetaCuration.xlsx', '2024_06_28_RepoMetaCuration.xlsx', '2024_07_20_RepoMetaCuration.xlsx', '2024_07_26_RepoMetaCuration.xlsx', '2024_07_31_RepoMetaCuration.xlsx', '2024_12_20_RepoMetaCuration.xlsx', '2025_01_23_RepoMetaCuration.xlsx', '2025_03_19_RepoMetaCuration.xlsx', '2025_03_25_RepoMetaCuration.xlsx', '2025_04_18_RepoMetaCuration.xlsx', '2025_04_24_RepoMetaCuration.xlsx', '2025_04_25_RepoMetaCuration.xlsx', '2025_05_14_RepoMetaCuration.xlsx', '2025_09_29_RepoMetaCuration.xlsx', '2025_09_30_RepoMetaCuration.xlsx', '2025_11_25_RepoMetaCuration.xlsx', '2025_12_04_RepoMetaCuration.xlsx', '2026_04_08_RepoMetaCuration.xlsx', '2026_04_13_RepoMetaCuration.xlsx', '2026_06_25_SourceMetaCuration.xlsx']


In [4]:
def clean_rawdf(df_raw):
    df = df_raw.fillna(-1)
    df.rename(columns={'type':'@type'}, inplace=True)
    return df


def clean_nones(a_dict):
    for k,v in list(a_dict.items()):
        if v == -1:
            del a_dict[k]
        if v == 'None':
            del a_dict[k]
        if v == None:
            del a_dict[k]
        if not isinstance(v,str) and not isinstance(v,dict) and not isinstance(v,list):
            if isnan(v):
                del a_dict[k]
    return a_dict


def clean_dict_array(dict_array):
    for eachdict in dict_array:
        eachdict = clean_nones(eachdict)
    return dict_array

### Process the resource_base sheet

In [5]:
def format_boolean(boolean_field):
    if boolean_field in [True, False]:
        return boolean_field
    if isinstance(boolean_field, str):
        clean_bool = boolean_field.strip().lower()
        if clean_bool in ['true', '1', 'yes']:
            return True
        if clean_bool in ['false', '0', 'no']:
            return False
    if boolean_field in [1, 1.0]:
        return True
    if boolean_field in [0, 0.0]:
        return False
    return boolean_field

def format_date(datefield):
    if isinstance(datefield, str):
        cleandate = datefield
    if isinstance(datefield, datetime):
        cleandate = datefield.strftime('%Y-%m-%d')
    return cleandate

def add_type(df):
    df['@type'] = 'nde:ResourceCatalog'
    return df

def run_quick_clean(df):
    empty_cols = [col for col in df.columns if df[col].isna().all()]
    df.drop(columns=empty_cols, inplace=True)
    df = df.fillna('None')
    if 'License type' in df.columns:
        df.drop('License type', axis=1, inplace=True)
    boolprops = ['hasAPI', 'isAccessibleForFree']
    for eachprop in boolprops:
        if eachprop in df.columns:
            df[eachprop] = df.apply(lambda row: format_boolean(row[eachprop]), axis=1)
    if 'genre' in df.columns:
        df['genre'] = df['genre'].astype(str).replace('generalist', 'Generalist')
    print(df)
    return df

In [6]:
filepath = os.path.join(data_path, filelist[-1])
df_base = pd.read_excel(filepath, 'resource_base', engine='openpyxl')
df_clean = add_type(run_quick_clean(df_base))
print(df_clean.head(n=2))
check = df_clean.iloc[0]['url']

                                                 name  \
0                            accessclinicaldata@NIAID   
1                                           ClinEpiDB   
2                                 COVID RADx Data Hub   
3   ImmPort - Bioinformatics For the Future of Imm...   
4                                          MalariaGEN   
5                                             MassIVE   
6                                            NCBI GEO   
7                  NICHD Data and Specimen Hub (DASH)   
8                                   Protein Data Bank   
9                                               Qiita   
10           The Database of Genotypes and Phenotypes   
11                   The Network Data Exchange (NDEx)   
12                                          VDJServer   
13                                           AmoebaDB   
14                                          bio.tools   
15                                           CryptoDB   
16                             

In [7]:
check = df_clean.iloc[0]['url']
print(check)

https://accessclinicaldata.niaid.nih.gov/


### Process collectionSize

In [8]:
def clean_minvalue(value):
    if isinstance(value, float) and value.is_integer():
        return int(value)
    return value

def create_collection_dict(df_collection):
    collection_dict = {}
    url_list = df_collection['url'].unique().tolist()
    df_collection['@type'] = 'PropertyValue'
    df_collection['minValue'] = df_collection['minValue'].apply(lambda x: clean_minvalue(x))
    for eachurl in url_list:
        tmpdf = df_collection.loc[df_collection['url']==eachurl].copy()
        tmpdf.drop('url', inplace=True, axis=1)
        tmp_array = tmpdf.to_dict(orient='records')
        collection_dict[eachurl] = clean_dict_array(tmp_array)
    return collection_dict

In [9]:
df_collection_raw = pd.read_excel(filepath, 'collectionSize', engine='openpyxl')
df_collection = clean_rawdf(df_collection_raw)
collection_dict = create_collection_dict(df_collection)
print(collection_dict[check])

[{'minValue': 12, 'unitText': 'Dataset', '@type': 'PropertyValue'}]


### Process the definedTerms sheet

In [10]:
def generate_dt_dict(df_dt):
    dfdt_dict = {}
    urlist = df_dt['url'].tolist()
    for eachurl in urlist:
        prop_dict = {}
        tmpdf = df_dt.loc[df_dt['url']==eachurl].copy()
        tmpdf.drop('url', inplace=True, axis=1)
        proplist = tmpdf['property'].unique().tolist()
        for eachprop in proplist:
            if eachprop == 'about':
                aboutdf = tmpdf[['name','prop.url','prop.description']].loc[tmpdf['property']=='about']
                aboutdf.rename(columns={'prop.url':'url','prop.description':'description'}, inplace=True)
                aboutdf[['@type']] = 'DefinedTerm'
                aboutlist = aboutdf.to_dict(orient='records')
                cleanabout = [clean_nones(x) for x in aboutlist]
                prop_dict['about'] = cleanabout
            else:
                prop_dict[eachprop] = tmpdf.loc[tmpdf['property']==eachprop]['prop.url'].unique().tolist()
        dfdt_dict[eachurl] = prop_dict
    return dfdt_dict

In [11]:
df_definedTerm_raw = pd.read_excel(filepath, 'definedTerms', engine='openpyxl')
df_dt = clean_rawdf(df_definedTerm_raw)
dfdt_dict = generate_dt_dict(df_dt)
print(dfdt_dict[check])

{'infectiousAgent': ['https://www.uniprot.org/taxonomy/2697049', 'https://www.uniprot.org/taxonomy/694009'], 'measurementTechnique': ['http://purl.obolibrary.org/obo/NCIT_C82639', 'http://purl.obolibrary.org/obo/NCIT_C15417', 'http://purl.obolibrary.org/obo/NCIT_C15228', 'http://purl.obolibrary.org/obo/NCIT_C66959', 'http://purl.obolibrary.org/obo/NCIT_C49659'], 'healthCondition': ['http://purl.obolibrary.org/obo/MONDO_0100096', 'http://purl.obolibrary.org/obo/MONDO_0018076', 'http://purl.obolibrary.org/obo/NCIT_C35650', 'http://purl.obolibrary.org/obo/MONDO_0005109', 'http://purl.obolibrary.org/obo/MONDO_0005249', 'http://purl.obolibrary.org/obo/NCIT_C154635', 'http://purl.obolibrary.org/obo/MONDO_0005550', 'http://purl.obolibrary.org/obo/MONDO_0000775'], 'species': ['https://www.uniprot.org/taxonomy/9606'], 'topicCategory': ['http://edamontology.org/topic_3324', 'http://edamontology.org/topic_3379', 'http://edamontology.org/topic_3305', 'http://edamontology.org/topic_3303', 'http://e

### Process the spatialTemporalCoverage sheet

In [12]:
def create_spatiotemporalcoverage_dict(df):
    coverage_dict = {}
    if len(df) != 0:
        dateprops = ['startDate', 'endDate']
        for eachprop in dateprops:
            if eachprop in list(df.columns.values):
                df[eachprop] = df.apply(lambda row: format_date(row[eachprop]) if row[eachprop] != -1 else -1, axis=1)
        urlist = df['url'].unique().tolist()
        for eachurl in urlist:
            prop_dict = {}
            tmpdf = df.loc[df['url']==eachurl].copy()
            for eachprop in tmpdf['property'].unique().tolist():
                propdf = tmpdf.loc[tmpdf['property']==eachprop].copy()
                propdf.drop(['url','property'], inplace=True, axis=1)
                tmp_array = propdf.to_dict(orient='records')
                tmp_array = clean_dict_array(tmp_array)
                if len(tmp_array) == 1:
                    prop_dict[eachprop] = tmp_array[0]
                else:
                    prop_dict[eachprop] = tmp_array
            coverage_dict[eachurl] = prop_dict
    return coverage_dict

In [13]:
df_coverage_raw = pd.read_excel(filepath, 'spatialTemporalCoverage', engine='openpyxl')
df_coverage = clean_rawdf(df_coverage_raw)
coverage_dict = create_spatiotemporalcoverage_dict(df_coverage)

try:
    print(coverage_dict[check])
except:
    print('no spatialTemporalCoverage data')

KeyError: 'property'

### Assemble the json records

In [ ]:
def process_records(filepath, context_dict):
    batchlist = []
    df_base = pd.read_excel(filepath, 'resource_base', engine='openpyxl')
    df_clean = add_type(run_quick_clean(df_base))
    today = datetime.now()
    df_clean['date'] = today.strftime('%Y-%m-%d')
    df_collection_raw = pd.read_excel(filepath, 'collectionSize', engine='openpyxl')
    df_collection = clean_rawdf(df_collection_raw)
    collection_dict = create_collection_dict(df_collection)
    df_definedTerm_raw = pd.read_excel(filepath, 'definedTerms', engine='openpyxl')
    df_dt = clean_rawdf(df_definedTerm_raw)
    dfdt_dict = generate_dt_dict(df_dt)
    df_coverage_raw = pd.read_excel(filepath, 'spatialTemporalCoverage', engine='openpyxl')
    df_coverage = clean_rawdf(df_coverage_raw)
    coverage_dict = create_spatiotemporalcoverage_dict(df_coverage)
    base_dict_array = df_clean.to_dict(orient='records')
    base_dict_array = clean_dict_array(base_dict_array)
    for eachdict in base_dict_array:
        url = eachdict['url']
        eachdict['@context'] = context_dict
        eachdict['includedInDataCatalog'] = {'@type':'DataCatalog','name':'Data Discovery Engine','url':'https://discovery.biothings.io/'}
        try:
            eachdict['collectionSize'] = collection_dict[url]
        except:
            pass
        try:
            eachdict.update(coverage_dict[url])
        except:
            pass
        try:
            eachdict.update(dfdt_dict[url])
        except:
            pass
        batchlist.append(eachdict)
    return batchlist

In [ ]:
context_dict = {'owl': 'http://www.w3.org/2002/07/owl#',
                'rdf': 'http://www.w3.org/1999/02/22-rdf-syntax-ns#',
                'rdfs': 'http://www.w3.org/2000/01/rdf-schema#',
                'schema': 'http://schema.org/',
                'niaid': 'https://discovery.biothings.io/ns/niaid/',
                'nde': 'https://discovery.biothings.io/ns/nde/'}

In [ ]:
batchlist = process_records(filepath, context_dict)
today = datetime.now()
with open(os.path.join(result_path, f'{today.strftime("%Y-%m-%d")}_batch_file.json'), 'w') as outfile:
    outfile.write(json.dumps(batchlist, indent=4))
print(batchlist[0]['name'])